In [ ]:
# collect the activity
ids = [ei for ei in range(execution_id)]

# get decision activities for all trials 
dec_acts = np.array([
    np.squeeze(model.parameters.results.get(ei))
    for ei in ids
])

In [ ]:
def compute_rt(act, threshold=.9):
    """compute reaction time
    take the activity of the decision layer...
    RT := the earliest time point when activity > threshold...
    """
    n_time_steps_, N_UNITS_ = np.shape(act)
    tps_pass_threshold = np.where(act[:, 0] > threshold)[0]
    if len(tps_pass_threshold) > 0:
        return tps_pass_threshold[0]
    return n_time_steps_

In [ ]:
# re-organize RT data
threshold = .9
rts = np.zeros((n_demand_levels, n_tasks, n_conditions))
counter = 0
for did, demand in enumerate(demand_levels):
    for tid, task in enumerate(TASKS):
        for cid, cond in enumerate(CONDITIONS):
            rts[did, tid, cid] = compute_rt(
                dec_acts[counter], threshold=threshold
            )
            counter += 1

### Plotting Task Demand & RT

The two figures created by the following cell qualitatively replicate Fig 13A (left) and Fig 13B (right) from <a href="https://www.ncbi.nlm.nih.gov/pubmed/2200075">Cohen et al. (1990)</a>.  The Y axis displays response time, and the X axis progresses upward in task demand unit activity.  (*Note that the key is consistent with previous figures in this notebook, but not all the conditions in the key are plotted.*) 

In the left panel we can see that Word reading (dashed black line) is generally faster than ink Color Naming (solid black line). The other prominent pattern is that increased activity in the task demand units leads to faster (lower) reaction times. 

The right panel compares Word reading under conflict (green dashed) to control (blue dashed).  It also displays part of the Color Naming plot under conflict (green solid).  These figures are truncated at the top to zoom in on key comparisons, and the full data extend well above the top of the Y-axis.    

These reaction times are in different units than human performance, but the overall trends make sense.  We can potentially use Task Demand as a way to model Attention/Effort. Increasing attention to the task improves performance yielding faster reaction times. 

In [ ]:
# plot prep
col_pal = sns.color_palette('colorblind', n_colors=3)
xticklabels = ['%.1f' % (d) for d in demand_levels]

f, axes = plt.subplots(1, 2, figsize=(13, 5))
# left panel
axes[0].plot(np.mean(rts[:, 0, :], axis=1), color='black', linestyle='-')
axes[0].plot(np.mean(rts[:, 1, :], axis=1), color='black', linestyle='--')
axes[0].set_title('RT as a function of task demand')
# axes[0].legend(TASKS, frameon=False, bbox_to_anchor=(.4, 1))
axes[0].legend(handles=lgd_elements, frameon=False, bbox_to_anchor=(.7, .95))
# right panel
clf_id = 1
n_skips = 2
axes[1].plot(np.arange(n_skips, n_demand_levels, 1),
             rts[n_skips:, 0, clf_id], color=col_pal[clf_id],
             label='conflicting word')
axes[1].plot(rts[:, 1, clf_id], color=col_pal[clf_id],
             linestyle='--', label='conflicting color')
axes[1].plot(rts[:, 1, 0], color=col_pal[0], linestyle='--', label='control')
axes[1].set_title('Compared the two conflict conditions')
# axes[1].legend(frameon=False, bbox_to_anchor=(.55, 1))
# common
axes[0].set_ylabel('Reaction time (RT)')
axes[1].set_ylim(axes[0].get_ylim())
for ax in axes:
    ax.set_xticks(range(n_demand_levels))
    ax.set_xticklabels(xticklabels)
    ax.set_xlabel('Demand')
f.tight_layout()
sns.despine()


Exercise 4. Interpret the task demand results above{exercise}
- Compare the results above with human performance in <a href="https://www.ncbi.nlm.nih.gov/pubmed/2200075">Cohen et al. (1990)</a> Figures 13A & 13B, and comment on a few interesting similarities and differences.  



Exercise 5: Putting it all together{exercise}

Note: Answers to this exercise can be qualitative and schematic -- you do not need to build the models (although you can if you like!), just describe how you would initially reason and plan to build them. 

5a.  How should task demand unit activity impact accuracy?

5b.  Concisely describe key elements of a model mimicking human performance that exhibits the appropriate influence of task demand activity on accuracy.

5c.  Describe steps that you could take, based on the models provided in this notebook, to build a model that monitors for conflict within trials and increases attention when conflict is present.  

5d.  Describe steps that you could take to build a model that monitors for errors after trials and increases attention on the subsequent trial.  


